Low-Cost Resume Optimization via Distilling Large LLM Behavior into a Fine-Tuned Small Language Model (SLM)

> **Project Overview:** This notebook shows installation of required packages necessary for dataset creation by extracting textual data from custom sourced 1800+ resumes(PDFs, Docx etc) and scrapes job descriptions from job posting websites and perform pre-processing by merging resume data with job description and data cleaning and used gemini batch api for processing resumes across job description for tailored resumes.

## 📋 Table of Contents

1. **Setup & Dependencies** - Install required packages and import libraries
2. **Dataset Creation** - Extract text from 1800+ resumes (PDF, DOCX)
3. **Job Scraper** - Scrape job descriptions from job posting websites
4. **Data Cleaning & Combination** - Merge resume data with job descriptions
5. **Gemini Batch API** - Process resumes at scale using Google's Gemini API
6. **Training Dataset Preparation** - Format data for fine-tuning


---

## 1. 📦 Setup & Dependencies

### 1.1 Installation Commands (Optional)
Run the cells below if packages are not already installed. These are commented out by default.

In [ ]:
# Optional: Install Google Chrome for web scraping (Linux only)
# !wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
# !apt-get install -y ./google-chrome-stable_current_amd64.deb

In [ ]:
# Optional: Install web scraping dependencies
# !pip install selenium webdriver-manager beautifulsoup4

  Using cached selenium-4.38.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached trio-0.32.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached websocket_client-1.9.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
  Using cached python_dotenv-1.2.1-py3-none-any.whl.metadata (25 kB)
  Using cached soupsieve-2.8-py3-none-any.whl.metadata (4.6 kB)
Using cached selenium-4.38.0-py3-none-any.whl (9.7 MB)
Using cached trio-0.32.0-py3-none-any.whl (512 kB)
Using cached trio_websocket-0.12.2-py3-none-any.whl (21 kB)
Using cached websocket_client-1.9.0-py3-none-any.whl (82 kB)
Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl (27 kB)
Using cach

In [ ]:
# Optional: Install all project dependencies
# !pip install pandas numpy tqdm requests PyPDF2 python-docx selenium webdriver-manager \
#              beautifulsoup4 lxml pywin32 pyarrow google-genai torch transformers \
#              datasets trl peft bitsandbytes accelerate

### 1.2 Import Libraries

Import all required libraries for data processing, web scraping, ML/AI, and file handling.

In [ ]:
# Standard library imports
import os
import json
import re
import time
import logging
import base64
from datetime import datetime
from pathlib import Path
from typing import Optional, Dict, Any, Tuple, List

# Data processing
import pandas as pd
import numpy as np
from tqdm import tqdm
import requests

# Document parsing
from PyPDF2 import PdfReader
from PyPDF2.errors import PdfReadError
from docx import Document
from docx.opc.constants import RELATIONSHIP_TYPE as RT
from docx.opc.exceptions import PackageNotFoundError

# Web scraping
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

# Google AI
from google import genai
from google.genai import types


print("✅ All libraries imported successfully!")

c:\ProgramData\anaconda3\envs\finetune\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## 2. 📄 Dataset Creation from 1800+ Resumes

This section extracts text content from resume documents in multiple formats (PDF, DOCX, DOC).

### Key Features:
- **PDF Extraction:** Uses PyPDF2 to extract text and detect embedded images
- **DOCX Extraction:** Parses Word documents and identifies image relationships  
- **DOC Extraction:** Uses Windows COM automation (requires pywin32 and Microsoft Word)
- **Output Format:** JSONL with filename, filetype, resume text, and image detection flag

### Configuration

In [ ]:
# Windows-specific imports for legacy .doc file support
try:
    import win32com.client as win32
    import pywintypes
    print("✅ pywin32 available - .doc file support enabled")
except ImportError:
    win32 = None
    pywintypes = None
    print("⚠️ pywin32 not available - .doc files will be skipped")

# Define recoverable errors that shouldn't stop processing
RECOVERABLE_EXTRACTION_ERRORS = (
    ValueError,
    OSError,
    PdfReadError,
    PackageNotFoundError,
    KeyError,
    RuntimeError,
)
# if pywintypes is not None and hasattr(pywintypes, "com_error"):
#     RECOVERABLE_EXTRACTION_ERRORS = RECOVERABLE_EXTRACTION_ERRORS + (
#         pywintypes.com_error,  # type: ignore[attr-defined]
#     )

In [ ]:
# Configuration: File paths and supported formats
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".doc"}
DEFAULT_INPUT_PATH = Path(r"C:\Users\VaishnavBusha\Documents\Repo\Colab-proj-2\data\resumes") 
DEFAULT_OUTPUT_PATH = Path("files\dataset\extracted-resume-data\resume_text.jsonl")

print(f"📁 Input path: {DEFAULT_INPUT_PATH}")
print(f"📁 Output path: {DEFAULT_OUTPUT_PATH}")
print(f"📄 Supported formats: {SUPPORTED_EXTENSIONS}")

### 2.1 Document Extraction Functions

The following code defines helper classes and functions for extracting text from various document formats.

In [ ]:
class WordAutomationClient:
    """Thin wrapper around Word COM automation to read legacy .doc files."""

    def __init__(self) -> None:
        if win32 is None:
            raise ImportError("pywin32 is required for .doc support on Windows")
        self._word = win32.Dispatch("Word.Application")
        self._word.Visible = False

    def close(self) -> None:
        if self._word is not None:
            self._word.Quit()
            self._word = None

    def __enter__(self) -> "WordAutomationClient":
        return self

    def __exit__(self, exc_type, exc, exc_tb) -> None:  # type: ignore[override]
        self.close()

    def extract_doc(self, file_path: Path) -> tuple[str, bool]:
        if self._word is None:
            raise RuntimeError("Word automation client is closed")
        document = self._word.Documents.Open(str(file_path))
        try:
            text = document.Content.Text
            contains_images = bool(document.InlineShapes.Count or document.Shapes.Count)
        finally:
            document.Close(False)
        return text.strip(), contains_images


def extract_text_from_pdf(file_path: Path) -> str:
    """Extract text from a PDF file."""
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += (page.extract_text() or "") + "\n"
    return text.strip()


def extract_text_from_docx(file_path: Path) -> str:
    """Extract text from a DOCX file."""
    doc = Document(file_path)
    text = ""
    for para in doc.paragraphs:
        text += para.text + "\n"
    return text.strip()


def extract_text(file_path: Path) -> str:
    """Extract text from a document file based on its extension."""
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return extract_text_from_pdf(file_path)
    if suffix == ".docx":
        return extract_text_from_docx(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def _xobject_dict_contains_images(x_objects) -> bool:
    if not x_objects:
        return False
    # Dereference x_objects if it's an indirect object
    try:
        x_objects = x_objects.get_object()
    except AttributeError:
        pass
    if not hasattr(x_objects, "values"):
        return False
    for obj in x_objects.values():
        try:
            x_obj = obj.get_object()
        except AttributeError:
            x_obj = obj
        subtype = x_obj.get("/Subtype")
        if subtype == "/Image":
            return True
        if subtype == "/Form":
            child_resources = x_obj.get("/Resources")
            child_x_objects = None
            if child_resources:
                # Dereference indirect objects
                try:
                    child_resources = child_resources.get_object()
                except AttributeError:
                    pass
                if hasattr(child_resources, "get"):
                    child_x_objects = child_resources.get("/XObject")
            if _xobject_dict_contains_images(child_x_objects):
                return True
    return False


def pdf_has_images(file_path: Path) -> bool:
    reader = PdfReader(file_path)
    for page in reader.pages:
        if hasattr(page, "images") and page.images:
            return True
        resources = page.get("/Resources")
        if not resources:
            continue
        # Dereference indirect objects
        try:
            resources = resources.get_object()
        except AttributeError:
            pass
        if not hasattr(resources, "get"):
            continue
        x_objects = resources.get("/XObject")
        if _xobject_dict_contains_images(x_objects):
            return True
    return False


def docx_has_images(file_path: Path) -> bool:
    doc = Document(file_path)
    return any(rel.reltype == RT.IMAGE for rel in doc.part.rels.values())


def has_images(file_path: Path) -> bool:
    suffix = file_path.suffix.lower()
    if suffix == ".pdf":
        return pdf_has_images(file_path)
    if suffix == ".docx":
        return docx_has_images(file_path)
    raise ValueError(f"Unsupported file type: {file_path.suffix}")


def collect_documents(target: Path) -> List[Path]:
    if target.is_file():
        return [target]
    if target.is_dir():
        return sorted(
            [f for f in target.rglob("*") if f.suffix.lower() in SUPPORTED_EXTENSIONS]
        )
    raise FileNotFoundError(f"'{target}' is not a valid file or directory.")


def process_document(
    file_path: Path, word_client: Optional["WordAutomationClient"]
) -> Dict[str, str | bool]:
    suffix = file_path.suffix.lower()
    if suffix == ".doc":
        if word_client is None:
            raise RuntimeError(
                ".doc support requires Microsoft Word and pywin32; both appear unavailable."
            )
        resume_text, contains_images = word_client.extract_doc(file_path)
    else:
        resume_text = extract_text(file_path)
        contains_images = has_images(file_path)

    return {
        "filename": file_path.name,
        "filetype": suffix,
        "resume_text": resume_text,
        "has_images": contains_images,
    }


def main() -> None:
    # Update these paths to control which resumes are processed and where JSONL output lands.
    input_path = DEFAULT_INPUT_PATH
    output_path = DEFAULT_OUTPUT_PATH

    try:
        documents = collect_documents(input_path)
    except FileNotFoundError as exc:
        print(exc)
        return

    if not documents:
        print(f"No PDF/DOC/DOCX files found under '{input_path}'.")
        return

    needs_word = any(path.suffix.lower() == ".doc" for path in documents)
    word_client: Optional[WordAutomationClient] = None
    if needs_word:
        try:
            word_client = WordAutomationClient()
        except ImportError as exc:
            print(f"Cannot open .doc files: {exc}")
            return

    processed_count = 0
    output_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with output_path.open("w", encoding="utf-8") as fh:
            for doc_path in documents:
                try:
                    record = process_document(doc_path, word_client)
                except RECOVERABLE_EXTRACTION_ERRORS as exc:
                    print(f"Skipping {doc_path}: {exc}")
                    continue
                fh.write(json.dumps(record, ensure_ascii=False) + "\n")
                processed_count += 1
                print(f"Processed {doc_path.name}")
    finally:
        if word_client is not None:
            word_client.close()

    if processed_count:
        print(
            f"Wrote metadata for {processed_count} file(s) to '{output_path.as_posix()}'"
        )
    else:
        print("No files were successfully processed; see logs above for details.")



In [ ]:
# Execute the resume extraction pipeline
if __name__ == "__main__":
    main()

---

## 3. 🔍 Job Scraper

This section scrapes job descriptions from job posting websites using Selenium WebDriver.

### Features:
- **Headless Browser:** Runs Chrome in headless mode for efficiency
- **Rate Limiting:** Configurable delays between requests to avoid blocking
- **Progress Tracking:** Real-time progress bar with success/failure counts
- **Incremental Saving:** Results saved after each scrape to prevent data loss

### 3.1 Network Check
First, verify internet connectivity by checking the external IP address.

In [ ]:
# Verify internet connectivity
try:
    response = requests.get('https://api64.ipify.org?format=json', timeout=10)
    response.raise_for_status()
    ip_address = response.json().get('ip')
    print(f"✅ Connected! Your IP Address: {ip_address}")
except requests.exceptions.RequestException as e:
    print(f"❌ Network error: {e}")

In [ ]:
# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

In [ ]:
class JobScraper:
    """Handles web scraping of job descriptions from various job posting websites."""

    def __init__(self, headless=True, wait_time=5, delay=2):
        """
        Initialize the job description scraper.

        Args:
            headless (bool): Run browser in headless mode
            wait_time (int): Time to wait for page to load (seconds)
            delay (int): Delay between requests (seconds)
        """
        self.headless = headless
        self.wait_time = wait_time
        self.delay = delay
        self.driver = None

    def setup_driver(self):
        """Set up Chrome WebDriver with appropriate options."""
        options = webdriver.ChromeOptions()
        if self.headless:
            options.add_argument('--headless')
        options.add_argument('--no-sandbox')
        options.add_argument('--disable-dev-shm-usage')
        options.add_argument('--disable-gpu')
        options.add_argument('--window-size=1920,1080')

        self.driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()),
            options=options
        )
        return self.driver

    def close_driver(self):
        """Close the WebDriver if it exists."""
        if self.driver:
            self.driver.quit()
            self.driver = None

    def scrape_job(self, job_url):
        """
        Extract job description from a job posting URL.

        Args:
            job_url (str): URL of the job posting

        Returns:
            tuple: (job_description, status)
        """
        if not self.driver:
            self.setup_driver()

        try:
            self.driver.get(job_url)
            time.sleep(self.wait_time)

            soup = BeautifulSoup(self.driver.page_source, 'html.parser')

            body_text = soup.body.get_text() if soup.body else ""

            # Extract the job description part
            # Assuming it starts after job details like Full-time, Onsite, etc.
            lines = body_text.split('\n')
            desc_start = False
            description = []

            for line in lines:
                line = line.strip()
                if any(keyword in line for keyword in ['Full-time', 'Onsite', 'Remote', 'Hybrid', 'Part-time']):
                    desc_start = True
                if desc_start and line:
                    description.append(line)

            job_text = ' '.join(description)

            if len(job_text) < 100:
                return None, "Description too short (< 100 chars)"

            # Truncate very long descriptions
            if len(job_text) > 10000:
                job_text = job_text[:10000] + "..."

            return job_text, "Success"

        except Exception as e:
            logger.error(f"Error scraping URL {job_url}: {str(e)}")
            return None, f"Error: {str(e)}"

    def scrape_from_csv(self, input_csv, output_csv=None):
        """Scrape jobs from CSV file with URLs."""

        if not os.path.exists(input_csv):
            logger.error(f"Input CSV file not found: {input_csv}")
            return None

        # Read input CSV
        try:
            df = pd.read_csv(input_csv)
        except Exception as e:
            logger.error(f"Error reading CSV: {e}")
            return None

        # Find URL column - look for 'Apply', 'url', or 'link' columns
        url_column = None

        # Priority order: Apply > URL > Link
        column_priorities = ['apply', 'url', 'link']

        for priority_col in column_priorities:
            for col in df.columns:
                if priority_col in col.lower():
                    url_column = col
                    break
            if url_column:
                break

        if url_column is None:
            logger.error("No URL column found in CSV. Expected column with 'Apply', 'URL', or 'Link' in name.")
            logger.info(f"Available columns: {list(df.columns)}")
            return None

        logger.info(f"Found {len(df)} URLs to scrape in column '{url_column}'")

        # Create output filename if not provided
        if output_csv is None:
            timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
            output_csv = f"data/scraped_jobs_{timestamp}.csv"
            os.makedirs("data", exist_ok=True)

        # Create CSV file with headers if it doesn't exist
        file_exists = os.path.exists(output_csv)
        if not file_exists:
            with open(output_csv, 'w', encoding='utf-8', newline='') as f:
                f.write('URL,Job Description,Scrape Status\n')

        # Scrape each URL
        successful = 0
        failed = 0

        try:
            with tqdm(total=len(df), desc="Scraping jobs") as pbar:
                for idx, row in df.iterrows():
                    job_url = row[url_column]

                    if pd.isna(job_url) or not job_url.strip():
                        # Save immediately to CSV
                        result_df = pd.DataFrame([[job_url, "", "Empty URL"]],
                                                columns=['URL', 'Job Description', 'Scrape Status'])
                        result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')
                        failed += 1
                        pbar.update(1)
                        continue

                    # Scrape job
                    description, status = self.scrape_job(job_url)

                    # Save immediately to CSV after each scrape
                    result_df = pd.DataFrame([[
                        job_url,
                        description if description else "",
                        status
                    ]], columns=['URL', 'Job Description', 'Scrape Status'])

                    result_df.to_csv(output_csv, mode='a', header=False, index=False, encoding='utf-8')

                    if status == "Success":
                        successful += 1
                    else:
                        failed += 1

                    pbar.set_postfix({
                        'Success': successful,
                        'Failed': failed,
                        'Rate': f"{successful/(successful+failed)*100:.1f}%" if (successful+failed) > 0 else "0%"
                    })

                    # Delay between requests
                    time.sleep(self.delay)
                    pbar.update(1)

            logger.info(f"Results saved to {output_csv}")
            logger.info(f"Summary: {successful} successful, {failed} failed")
            return output_csv,result_df

        except Exception as e:
            logger.error(f"Error during scraping: {e}")
            logger.info(f"Partial results saved to {output_csv}")
            logger.info(f"Summary before error: {successful} successful, {failed} failed")
            return output_csv,result_df
        finally:
            self.close_driver()

---

## 4. 🧹 Data Cleaning & Combination

This section processes and integrates the scraped job descriptions with the resume dataset.

### Pipeline Steps:
1. **Load Data:** Read scraped job data (CSV) and resume text data (JSONL)
2. **Filter Resumes:** Remove entries containing images (can't extract text reliably)
3. **Clean Text:** Remove non-ASCII characters and normalize whitespace
4. **Match Data:** Randomly assign job descriptions to resumes for training pairs
5. **Export:** Save combined dataset in multiple formats (CSV, Parquet, JSONL)

### 4.1 Configure Paths

In [ ]:
# Configuration for job scraping
input_csv = "files/job-scraper/job_links_merged.csv"
output_folder = 'files/job-scraper'

print(f"📁 Input CSV: {input_csv}")
print(f"📁 Output folder: {output_folder}")

In [ ]:
# Run the job scraping pipeline
os.makedirs(output_folder, exist_ok=True)

timestamp = datetime.now().strftime("%H%M%S%m%d%Y")
output_csv = os.path.join(output_folder, f"scraped_jobs_{timestamp}.csv")

# Initialize scraper and run
scraper = JobScraper(headless=True, wait_time=5, delay=2)
print("🚀 Starting job scraping...")

result_file = scraper.scrape_from_csv(input_csv, output_csv)

if result_file:
    print(f"✅ Scraping complete! Results saved to: {result_file}")
else:
    print("❌ Scraping failed!")

### 4.2 Load and Preview Data

In [ ]:
# Load cleaned job data and resume text data
df_jobs = pd.read_csv('files/job-scraper/cleaned_scraped_jobs_21390711212025.csv')
df_resumes = pd.read_json('files/dataset/extracted-resume-data/resume_text.jsonl', lines=True)

print(f"📊 Jobs loaded: {len(df_jobs)} records")
print(f"📊 Resumes loaded: {len(df_resumes)} records")

### 4.3 Clean Resume Data

Remove resumes with images and clean text by removing non-ASCII characters and extra whitespace.

In [ ]:
# Filter out resumes containing images
df_resumes = df_resumes[~(df_resumes['has_images'] == True)]
print(f"📊 Resumes after removing images: {len(df_resumes)}")

def clean_text(text: str) -> str:
    """Remove non-ASCII characters and normalize whitespace."""
    text = re.sub(r'[^\x00-\x7F]+', '', text)  # Remove non-ASCII
    text = re.sub(r'\s+', ' ', text).strip()    # Normalize whitespace
    return text

# Apply cleaning to resume text and filename
df_resumes['resume_text'] = df_resumes['resume_text'].apply(clean_text)
df_resumes['filename'] = df_resumes['filename'].apply(clean_text)

# Filter out very short resumes
df_resumes = df_resumes[df_resumes['resume_text'].str.len() > 10]
print(f"📊 Resumes after length filter: {len(df_resumes)}")

# Additional cleaning: remove special characters
df_resumes['resume_text'] = (df_resumes['resume_text']
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
)

print(f"✅ Resume data cleaned successfully!")

In [ ]:
# Preview the cleaned data
print("=" * 50)
print("RESUMES DATAFRAME INFO:")
print("=" * 50)
df_resumes.info()
print("\n" + "=" * 50)
print("JOBS DATAFRAME INFO:")
print("=" * 50)
df_jobs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 249 entries, 0 to 248
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   URL              249 non-null    object
 1   Job Description  249 non-null    object
 2   Scrape Status    249 non-null    object
dtypes: object(3)
memory usage: 6.0+ KB


### 4.4 Match Resumes with Job Descriptions

Randomly assign job descriptions to resumes to create training pairs for the model.

In [ ]:
# Create balanced job-resume pairs
job_descriptions = df_jobs['Job Description'].values
n_resumes = len(df_resumes)
n_jobs = len(job_descriptions)

# Evenly distribute job descriptions across resumes
repeats = (n_resumes // n_jobs) + 1
job_indices = np.tile(np.arange(n_jobs), repeats)[:n_resumes]

# Shuffle for randomization
np.random.seed(42)  # For reproducibility
np.random.shuffle(job_indices)

# Assign job descriptions
df_resumes['job_description'] = job_descriptions[job_indices]

# Display distribution statistics
print(f"📊 Total resumes: {len(df_resumes)}")
print(f"📊 Unique job descriptions: {df_resumes['job_description'].nunique()}")
print(f"\n📈 Distribution Statistics:")
print(f"   Min count per job: {df_resumes['job_description'].value_counts().min()}")
print(f"   Max count per job: {df_resumes['job_description'].value_counts().max()}")
print(f"   Mean count: {df_resumes['job_description'].value_counts().mean():.2f}")

Total resumes: 1565
Unique job descriptions: 245

Distribution of job descriptions (counts):
count    245.000000
mean       6.387755
std        0.882812
min        6.000000
25%        6.000000
50%        6.000000
75%        7.000000
max       13.000000
Name: count, dtype: float64

Min count: 6
Max count: 13

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
Index: 1565 entries, 0 to 2126
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1565 non-null   object
 1   filetype         1565 non-null   object
 2   resume_text      1565 non-null   object
 3   has_images       1565 non-null   bool  
 4   job_description  1565 non-null   object
dtypes: bool(1), object(4)
memory usage: 62.7+ KB


### 4.5 Export Combined Dataset

Save the combined dataset in multiple formats for flexibility.

In [ ]:
# Export to multiple formats
DATA_DIR = "./files/dataset/resume-data-with-job-description"
os.makedirs(DATA_DIR, exist_ok=True)

# Save in different formats
df_resumes.to_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df_resumes.to_json(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl'), orient='records', lines=True)

print("✅ Dataset exported to:")
print(f"   📄 {DATA_DIR}/final_resume_dataset.parquet")
print(f"   📄 {DATA_DIR}/final_resume_dataset.jsonl")

In [ ]:
# Verify exported files by comparing formats
DATA_DIR = "./files/dataset/resume-data-with-job-description"

# Read all three formats

df_parquet = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'))
df_jsonl = pd.read_json(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl'), lines=True)

# Compare shapes
print("=" * 50)
print("FORMAT COMPARISON")
print("=" * 50)
print(f"\n📊 Shape Comparison:")
print(f"   Parquet: {df_parquet.shape}")
print(f"   JSONL:   {df_jsonl.shape}")

# Compare file sizes
csv_size = os.path.getsize(os.path.join(DATA_DIR, 'final_resume_dataset.csv')) / (1024*1024)
parquet_size = os.path.getsize(os.path.join(DATA_DIR, 'final_resume_dataset.parquet')) / (1024*1024)
jsonl_size = os.path.getsize(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl')) / (1024*1024)

print(f"\n💾 File Size Comparison:")
print(f"   CSV:     {csv_size:.2f} MB")
print(f"   Parquet: {parquet_size:.2f} MB (most compact)")
print(f"   JSONL:   {jsonl_size:.2f} MB")


=== Shape Comparison ===
CSV:     (1565, 5)
Parquet: (1565, 5)
JSONL:   (1565, 5)

=== Columns Comparison ===
CSV columns:     ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
Parquet columns: ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']
JSONL columns:   ['filename', 'filetype', 'resume_text', 'has_images', 'job_description']

=== Data Types Comparison ===
CSV dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

Parquet dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

JSONL dtypes:
filename           object
filetype           object
resume_text        object
has_images           bool
job_description    object
dtype: object

=== File Size Comparison ===
CSV:     25.36 MB
Parquet: 9.48 MB
JSONL:   25.50 MB

=== Data Equality Check ===
C

,filename,filetype,resume_text,has_images,job_description
0,00_Willie_Ellis_Go_Python.docx,.docx,Willie Ellis Senior Software Engineer Buffalo ...,False,Hanger/Textiles jobs in United StatesOverviewC...
1,10272022 Resume.docx,.docx,SUMMARY Leverage my skills education and exper...,False,Production Associate - Garment Hanger/Inspecto...


In [ ]:
# Load the parquet file for subsequent processing (most efficient format)
DATA_DIR = "./files/dataset/resume-data-with-job-description"
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"📊 Loaded {len(df)} records from parquet file")

---

## 5. ☁️ Gemini Batch API Processing

This section uses **Google's Gemini API** for large-scale batch processing of resume generation requests.

### Advantages of Batch Processing:
- **Cost Efficient:** 50% discount on batch API requests
- **Scalable:** Process thousands of requests in parallel
- **Reliable:** Built-in retry and error handling

### 5.1 Define Prompts for Batch API

In [ ]:
# JSON Schema for tailored resume output
RESUME_SCHEMA = '''{"$schema":"http://json-schema.org/draft-04/schema#","type":"object","properties":{"personal_information":{"type":"object","properties":{"name":{"type":"string"},"email":{"type":"string"},"phone":{"type":"string"},"location":{"type":"string"},"socials":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]}},"required":["name","email","phone","location"]},"summary":{"type":"string"},"experiences":{"type":"array","items":[{"type":"object","properties":{"designation":{"type":"string"},"companyName":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"points":{"type":"array","items":[{"type":"string"}]}},"required":["designation","companyName","location","start_date"]}]},"education":{"type":"array","items":[{"type":"object","properties":{"institution":{"type":"string"},"degree":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"gpa":{"type":"string"}},"required":["institution","degree","location","start_date","gpa"]}]},"skills":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"data":{"type":"array","items":[{"type":"string"}]}},"required":["name","data"]}]},"projects":{"type":"array","items":[{"type":"object","properties":{"projectName":{"type":"string"},"caption":{"type":"string"},"location":{"type":"string"},"start_date":{"type":"string"},"end_date":{"type":"string"},"url":{"type":"string"},"projectDetails":{"type":"array","items":[{"type":"string"}]},"externalSources":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"link":{"type":"string"}},"required":["name","link"]}]},"technologiesUsed":{"type":"array","items":[{"type":"string"}]}},"required":["projectName","location","projectDetails"]}]},"certifications":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"issuing_organization":{"type":"string"},"issue_date":{"type":"string"},"expiration_date":{"type":"string"},"credential_id":{"type":"string"},"url":{"type":"string"}},"required":["name","issuing_organization","issue_date","expiration_date","credential_id","url"]}]},"awards":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"extracurricular_achievements":{"type":"array","items":[{"type":"object","properties":{"name":{"type":"string"},"type":{"type":"string"},"location":{"type":"string"},"date":{"type":"string"},"description":{"type":"string"}},"required":["name","type","location","date","description"]}]},"languages":{"type":"array","items":[{"type":"object","properties":{"language":{"type":"string"},"proficiency":{"type":"string"}},"required":["language","proficiency"]}]}},"required":["personal_information","education","skills","extracurricular_achievements"]}'''


SYSTEM_PROMPT = """You create a tailored resume based on the job description.

Your task:
1. Read the RESUME_TEXT.
2. Read the JOB_DESCRIPTION.
3. Use only the information inside these two.
4. Follow the SCHEMA exactly.
5. Write a tailored resume in JSON using the SCHEMA.
6. Do not output anything outside the JSON.
7. If a field is missing in the resume, write a short, safe placeholder that fits the job.

"""

PROMPT_TEMPLATE = """
RESUME_TEXT:
\"\"\"
<<PASTE RESUME HERE>>
\"\"\"

JOB_DESCRIPTION:
\"\"\"
<<PASTE JD HERE>>
\"\"\"

SCHEMA:
\"\"\"
<<PASTE JSON SCHEMA HERE>>
\"\"\"

Create a tailored resume in JSON following the SCHEMA exactly.
Use only content from the RESUME_TEXT but rewrite it to match the JOB_DESCRIPTION.
Do not add extra lines or explanation.
Output only JSON.
"""

In [ ]:
def _build_USER_prompt(resume_text: str, job_description: str) -> str:
    """Build the user prompt by filling in the template."""
    prompt = PROMPT_TEMPLATE.replace("<<PASTE RESUME HERE>>", resume_text)
    prompt = prompt.replace("<<PASTE JD HERE>>", job_description)
    prompt = prompt.replace("<<PASTE JSON SCHEMA HERE>>", RESUME_SCHEMA)
    return prompt

### 5.2 Generate Batch Request Files

Create JSONL files with batch requests for the Gemini API.

In [ ]:
# Load data and prepare batch requests
DATA_DIR = ".files/dataset/resume-data-with-job-description/"
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')

# Column names
resume_col = "resume_text"
job_col = "job_description"

def generate_request_file(batch_name: str, df_subset: pd.DataFrame) -> None:
    """Generate a JSONL file with batch requests for the Gemini API."""
    output_filename = f"batch_requests_{batch_name}.jsonl"
    output_path = Path(f"dataset/batch_requests/{output_filename}")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    requests = []
    print(f"\n🔄 Generating requests for: {output_filename}")

    for idx, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc=f"Processing {batch_name}"):
        user_content = _build_USER_prompt(row[resume_col], row[job_col])

        request_entry = {
            "key": f"request-{batch_name}-idx{idx}",
            "request": {
                "contents": [{"role": "user", "parts": [{"text": user_content}]}],
                "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
                "generationConfig": {"responseMimeType": "application/json", "temperature": 0.2}
            }
        }
        requests.append(request_entry)

    # Write to JSONL
    with output_path.open('w', encoding='utf-8') as f:
        for req in requests:
            f.write(json.dumps(req) + "\n")

    print(f"✅ Created {output_filename} with {len(requests)} requests")

# Generate batch file (adjust indices as needed)
batch_key = "batch-4"
df_subset = df.iloc[521:1565]
generate_request_file(batch_key, df_subset)


Generating requests for  -> batch_requests_batch-4.jsonl


Processing batch-4: 100%|██████████| 1044/1044 [00:00<00:00, 11106.29it/s]

Success: Created batch_requests_batch-4.jsonl with 1044 requests.


In [ ]:
# Verify dataset info
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1565 entries, 0 to 2126
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1565 non-null   object
 1   filetype         1565 non-null   object
 2   resume_text      1565 non-null   object
 3   has_images       1565 non-null   bool  
 4   job_description  1565 non-null   object
dtypes: bool(1), object(4)
memory usage: 62.7+ KB


### 5.3 Configure Gemini API Client

Set up authentication for the Google GenAI client.

#### Option 1: Configure with Gemini API key

In [ ]:
# Option 1: Configure with Gemini API key
client = genai.Client(api_key="YOUR_API_KEY_HERE")
print("✅ GenAI configured with API key!")

# Note: Replace with your actual API key or use environment variable
print("⚠️ API key configuration required - update this cell with your key")

GenAI configured successfully with gemini api key!


#### Option 2: Configure with Vertex AI (Google Cloud)

In [ ]:
# Option 2: Configure with Vertex AI (Google Cloud)
# Set the path to your service account key file
CREDENTIALS_PATH = "gen-lang-client-0465995346-fd97340f8167.json"

if os.path.exists(CREDENTIALS_PATH):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.path.abspath(CREDENTIALS_PATH)
    print(f"✅ Service account credentials loaded from: {CREDENTIALS_PATH}")
else:
    print(f"⚠️ Credentials file not found: {CREDENTIALS_PATH}")

In [ ]:
# Initialize Vertex AI client
client = genai.Client(
    vertexai=True,
    project='gen-lang-client-0465995346',
    location='us-central1'
)
print("✅ GenAI configured with Vertex AI!")

GenAI configured successfully with vertex api key!


### 5.4 Submit and Monitor Batch Jobs

In [ ]:
# Define batch file path
batch_file = 'files/dataset/batch_requests/batch_requests_batch-4[521-1535].jsonl'
print(f"📄 Batch file: {batch_file}")

In [ ]:
# Upload batch file to Google Cloud
uploaded_file = client.files.upload(
    file=batch_file,
    config=types.UploadFileConfig(display_name='my-batch-requests', mime_type='jsonl')
)
print(f"✅ Uploaded file: {uploaded_file.name}")

# Reference: Previous uploads
# batch-1: files/igujaoz0own5 → batches/mpfcsvvz5f3demju35es8w4tlqzunsjncxhn
# batch-4: files/saw7wfyrzg3p

In [ ]:
# Configure batch job parameters  for vertex AI
display_name = 'batch-upload-job-batch_requests_batch-4'
file_name = 'gs://harsha-dump/batch_requests_batch-4.jsonl'

print(f"📋 Job name: {display_name}")
print(f"📁 Source file: {file_name}")

In [ ]:
# Create and submit batch job
file_batch_job = client.batches.create(
    model="gemini-2.5-flash",
    src=file_name,
    config={'display_name': display_name}
)
print(f"✅ Created batch job: {file_batch_job.name}")

Created batch job: projects/982088254342/locations/us-central1/batchPredictionJobs/8949234702531690496


In [ ]:
# Check batch job status
job_name = "projects/982088254342/locations/us-central1/batchPredictionJobs/2288129378674016256"

completed_states = {'JOB_STATE_SUCCEEDED', 'JOB_STATE_FAILED', 'JOB_STATE_CANCELLED', 'JOB_STATE_EXPIRED'}

print(f"🔍 Checking status for: {job_name}")
batch_job = client.batches.get(name=job_name)
print(f"📊 Current state: {batch_job.state.name}")

if batch_job.state.name == 'JOB_STATE_FAILED':
    print(f"❌ Error: {batch_job.error}")

Polling status for job: projects/982088254342/locations/us-central1/batchPredictionJobs/2288129378674016256
Current state: JOB_STATE_RUNNING
Current state: JOB_STATE_RUNNING


In [ ]:
# Cancel batch job if needed
# client.batches.cancel(name=job_name)
# print("⚠️ Batch job cancelled")

In [ ]:
# Download batch results when job completes
job_name = "batches/mpfcsvvz5f3demju35es8w4tlqzunsjncxhn"
batch_job = client.batches.get(name=job_name)

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    if batch_job.dest and batch_job.dest.file_name:
        result_file_name = batch_job.dest.file_name
        print(f"📥 Downloading results from: {result_file_name}")

        file_content = client.files.download(file=result_file_name)

        # Save to local file
        output_path = Path(f"files/dataset/batch_results/{result_file_name.replace('/', '_')}.jsonl")
        output_path.parent.mkdir(parents=True, exist_ok=True)

        with output_path.open('wb') as f:
            f.write(file_content)

        print(f"✅ Results saved to: {output_path}")
    else:
        print("⚠️ No file results found")
else:
    print(f"⚠️ Job state: {batch_job.state.name}")

### 5.5 Process Batch Results

Parse and clean the batch API responses, then merge with the original dataset.

In [ ]:
# Add ID column for tracking
df['id'] = df.index
print(f"✅ Added ID column to dataframe")

In [ ]:
# Save and reload dataframe with ID
DATA_DIR = ".\files\dataset\resume-data-with-job-description"
df.to_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"✅ Dataframe saved and reloaded: {len(df)} records")

In [ ]:
# Load batch results from multiple files
batch_result_files = [
    './files/dataset/batch_results/files_batch-mpfcsvvz5f3demju35es8w4tlqzunsjncxhn.jsonl',
    './files/dataset/batch_results/batch-output_prediction-model-2025-12-02T21_47_49.533170Z_predictions.jsonl'
]

list_of_dfs = []
for path in batch_result_files:
    if os.path.exists(path):
        temp = pd.read_json(path, lines=True)
        list_of_dfs.append(temp)
        print(f"📄 Loaded {len(temp)} records from: {os.path.basename(path)}")

# Combine all results
results_df = pd.concat(list_of_dfs, ignore_index=True)
results_df = results_df.drop(columns=['request', 'status', 'processed_time'], errors='ignore')

print(f"\n📊 Total combined records: {len(results_df)}")
results_df.info()

Loaded 521 records from first batch result
Loaded 1044 records from second batch result
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1565 entries, 0 to 1564
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   response  1565 non-null   object
 1   key       1565 non-null   object
dtypes: object(2)
memory usage: 24.6+ KB


,response,key
0,"{'candidates': [{'index': 0, 'finishReason': '...",request-batch-1-idx0-1
1,"{'responseId': 'FEUvaf3SLsnYqtsPka-dGA', 'usag...",request-batch-1-idx1-2
2,"{'candidates': [{'index': 0, 'finishReason': '...",request-batch-1-idx2-3
3,"{'responseId': 'FUUvabHPMOD6qtsPicKv-QY', 'mod...",request-batch-1-idx3-4
4,"{'candidates': [{'finishReason': 'STOP', 'cont...",request-batch-1-idx6-5


In [ ]:
def extract_indices(key: str) -> Optional[int]:
    """Extract the original index from the batch request key."""
    pattern = re.compile(r'idx(\d+)')
    match = pattern.search(key)
    return int(match.group(1)) if match else None

In [ ]:
def _extract_json(response: str) -> Optional[Dict[str, Any]]:
    """Extract JSON from model response with multiple fallback strategies."""
    if not response:
        return None

    response = response.strip()

    # Strategy 1: Direct parsing
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        pass

    # Strategy 2: Extract from ```json code blocks
    if "```json" in response:
        start = response.find("```json") + 7
        end = response.find("```", start)
        if end > start:
            try:
                return json.loads(response[start:end].strip())
            except json.JSONDecodeError:
                pass

    # Strategy 3: Extract from generic code blocks
    if "```" in response:
        start = response.find("```") + 3
        newline_pos = response.find("\n", start)
        if newline_pos > start:
            start = newline_pos + 1
        end = response.find("```", start)
        if end > start:
            try:
                return json.loads(response[start:end].strip())
            except json.JSONDecodeError:
                pass

    # Strategy 4: Extract between curly braces
    start = response.find("{")
    end = response.rfind("}") + 1
    if start >= 0 and end > start:
        try:
            return json.loads(response[start:end])
        except json.JSONDecodeError:
            pass

    return None

In [ ]:
# Process batch results and extract tailored resumes
processed_results = []
errors = []

for idx, row in tqdm(results_df.iterrows(), total=len(results_df), desc="Processing results"):
    key = row['key']
    extracted_index = extract_indices(key)

    result = {
        'key': key,
        'original_index': extracted_index,
        'status': 'success',
        'tailored_resume': None,
        'raw_response': None,
        'error': None,
        'found_in_df': extracted_index in df.index.tolist() if extracted_index else False
    }

    try:
        response = row.get('response', {})
        candidates = response.get('candidates', [])

        if not candidates:
            result['status'] = 'no_candidates'
            result['error'] = 'No candidates in response'
        else:
            parts = candidates[0].get('content', {}).get('parts', [])
            if not parts:
                result['status'] = 'no_parts'
                result['error'] = 'No parts in content'
            else:
                text = parts[0].get('text', '')
                result['raw_response'] = text

                cleaned_json = _extract_json(text)
                if cleaned_json:
                    result['tailored_resume'] = cleaned_json
                else:
                    result['status'] = 'json_parse_error'
                    result['error'] = 'Failed to parse JSON'
    except Exception as e:
        result['status'] = 'error'
        result['error'] = str(e)
        errors.append({'key': key, 'error': str(e)})

    processed_results.append(result)

processed_df = pd.DataFrame(processed_results)

# Display summary
print(f"\n" + "=" * 50)
print("PROCESSING SUMMARY")
print("=" * 50)
print(f"Total records: {len(processed_df)}")
print(f"\n📊 Status Distribution:")
print(processed_df['status'].value_counts())
print(f"\n✅ Found in original df: {processed_df['found_in_df'].sum()}")

Mapping original indices: 100%|██████████| 1565/1565 [00:00<00:00, 8792.20it/s]


=== Processing Summary ===
Total records: 1565
Status distribution:
status
success             1530
json_parse_error      22
no_parts              13
Name: count, dtype: int64

Found in original df: 1565
Not found in original df: 0


In [ ]:
# Count successful extractions
successful = processed_df[processed_df['status'] == 'success']
print(f"✅ Successful JSON extractions: {len(successful)}")

Successful extractions: 1530

=== Sample Tailored Resume ===
Key: request-batch-1-idx0-1
Original Index: 0

Tailored Resume JSON (first 500 chars):
{
  "personal_information": {
    "name": "Willie Ellis",
    "email": "willieellis0177gmailcom",
    "phone": "760 995 2578",
    "location": "Buffalo New York",
    "socials": [
      {
        "name": "LinkedIn",
        "link": "httpswwwlinkedincominwillieellisa6b492212"
      }
    ]
  },
  "summary": "Highly motivated professional with 7 years of work experience, demonstrating strong commitment, dependability, and hard work. Proven ability to implement process improvements to optimize effi...


In [ ]:
# Merge tailored resumes back to original dataframe
df['tailored_resume'] = None
count = 0
not_found = 0

for idx, row in tqdm(successful.iterrows(), total=len(successful), desc="Merging results"):
    original_idx = row['original_index']

    if original_idx in df['id'].values:
        df.loc[df['id'] == original_idx, 'tailored_resume'] = [row['tailored_resume']]
        count += 1
    else:
        not_found += 1

print(f"\n" + "=" * 50)
print("MERGE SUMMARY")
print("=" * 50)
print(f"✅ Successfully merged: {count}")
print(f"⚠️ Not found in df: {not_found}")
print(f"📊 Total with tailored_resume: {df['tailored_resume'].notna().sum()}")

# Remove rows without tailored resumes
print(f"\n📊 Before filtering: {len(df)} rows")
df = df[df['tailored_resume'].notna()]
df = df.drop(columns=['has_images'], errors='ignore')
print(f"📊 After filtering: {len(df)} rows")

Mapping tailored resumes to df: 100%|██████████| 1530/1530 [00:00<00:00, 2359.29it/s]


=== Mapping Summary ===
Successfully mapped: 1530
Not found in df: 0
Total rows with tailored_resume: 1530
Before removing nulls: 1565 rows
After removing nulls: 1530 rows


### 5.6 Save Final Dataset

Export the combined dataset with tailored resumes.

In [ ]:
# Save final dataset in multiple formats
DATA_DIR = "./files/dataset/resume-data-with-job-description"

df.to_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow', compression='snappy')
df.to_json(os.path.join(DATA_DIR, 'final_resume_dataset.jsonl'), orient='records', lines=True, force_ascii=False)
df.to_csv(os.path.join(DATA_DIR, 'final_resume_dataset.csv'), index=False, encoding='utf-8-sig')

print("✅ Final dataset saved:")
print(f"   📄 {DATA_DIR}/final_resume_dataset.parquet")
print(f"   📄 {DATA_DIR}/final_resume_dataset.jsonl")
print(f"   📄 {DATA_DIR}/final_resume_dataset.csv")

# Reload for verification
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"\n📊 Verified: {len(df)} records loaded")

---

## 6. 📝 Training Dataset Preparation

Convert the processed data into the format required for fine-tuning (chat-style JSONL).

### Dataset Format:
```json
{
  "messages": [
    {"role": "system", "content": "..."},
    {"role": "user", "content": "..."},
    {"role": "assistant", "content": "{...JSON resume...}"}
  ]
}
```

In [ ]:
# Load the final dataset
DATA_DIR = "./files/dataset/resume-data-with-job-description"
df = pd.read_parquet(os.path.join(DATA_DIR, 'final_resume_dataset.parquet'), engine='pyarrow')
print(f"📊 Loaded {len(df)} records for training data preparation")

In [ ]:
# Verify dataframe structure
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1530 entries, 0 to 2126
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   filename         1530 non-null   object
 1   filetype         1530 non-null   object
 2   resume_text      1530 non-null   object
 3   job_description  1530 non-null   object
 4   id               1530 non-null   int64 
 5   tailored_resume  1530 non-null   object
dtypes: int64(1), object(5)
memory usage: 83.7+ KB


### 6.1 Convert to Chat Format

In [ ]:
# Convert dataset to chat format for fine-tuning
results = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Converting to chat format"):
    user_content = _build_USER_prompt(row["resume_text"], row["job_description"])
    resume = row["tailored_resume"]

    # Handle numpy arrays and other non-string types
    if isinstance(resume, np.ndarray):
        resume = resume.tolist()
        if len(resume) == 1:
            resume = resume[0]
    if not isinstance(resume, str):
        resume = str(resume)

    result = {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
            {"role": "assistant", "content": resume}
        ]
    }
    results.append(result)

# Save training dataset
output_filename = "final_training_dataset"
output_path = Path(f"dataset/training/{output_filename}.jsonl")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open('w', encoding='utf-8') as f:
    for req in results:
        f.write(json.dumps(req, ensure_ascii=False) + "\n")

print(f"✅ Created {output_filename}.jsonl with {len(results)} training examples")

Extracting tailored resumes: 100%|██████████| 1530/1530 [00:00<00:00, 2825.50it/s]



Success: Created final_training_dataset with 1530 requests.


### 6.2 Clean Dataset (Fix JSON Formatting)

In [ ]:
import json
import re

def clean_python_to_json(text):
    # Replace Python None/True/False
    text = text.replace("None", "null")
    text = text.replace("True", "true")
    text = text.replace("False", "false")

    # Convert single quotes to double quotes carefully
    text = re.sub(r"'", '"', text)

    # Remove numpy arrays
    text = re.sub(r'array\((.*?)\)', r'\1', text)

    # Remove dtype info
    text = re.sub(r'dtype=object', '', text)

    # Remove trailing commas before }
    text = re.sub(r',\s*}', '}', text)

    # Remove trailing commas before ]
    text = re.sub(r',\s*\]', ']', text)

    return text


cleaned_dataset = []

with open("files/dataset/training/final_training_dataset.jsonl", "r") as f:
    for line in f:
        item = json.loads(line)

        # Clean assistant message
        assistant_msg = item["messages"][-1]["content"]
        cleaned = clean_python_to_json(assistant_msg)

        # Replace assistant content
        item["messages"][-1]["content"] = cleaned

        cleaned_dataset.append(item)

# Write new clean dataset
with open("files/dataset/training/final_training_dataset_cleaned.jsonl", "w") as f:
    for item in cleaned_dataset:
        f.write(json.dumps(item) + "\n")

print("Dataset cleaned! Output saved to final_training_dataset_cleaned.jsonl")

Dataset cleaned! Output saved to train_clean.jsonl
